# microWakeWord — Train any wake word

A single-notebook trainer for custom wake words on ESPHome `micro_wake_word` devices
(M5Stack Atom Echo, Voice PE, etc).

Two modes:
- **`generate`** — Piper TTS generates ~30k positive samples + your confusables in Colab.
  Easiest path. Requires an IPA pronunciation of your wake word.
- **`bundle`** — You upload a zip with your own samples (real recordings, ElevenLabs
  voices, accent-matched TTS). Higher quality but you do the prep work.

## Required runtime
**Runtime → Change runtime type → A100 GPU + High-RAM**.
T4 OOMs during validation. A100 + High-RAM gives 40 GB VRAM + 85 GB system RAM.

## What you get
A `<output_name>.tflite` (~60 KB) + companion `.json` manifest, ready to drop into
your ESPHome config under `micro_wake_word: models:`.

## Workflow
1. Edit the **CONFIGURE HERE** cell below for your wake word
2. **Runtime → Run all**, walk away ~45 minutes
3. Find `<output_name>.tflite` + `.json` in your Drive folder when it's done
4. Test on hardware — likely needs manifest tuning (cutoff, sliding_window) for
   your specific model. See the deployment notes at the end.

Built from working production deployment of "Hey Harold" — all known
upstream bugs are patched in this notebook.


In [1]:
from google.colab import files
uploaded = files.upload()

Saving mira_user_voice_samples.zip to mira_user_voice_samples.zip


In [2]:
# === MIRA USER VOICE — EXTRACT + CHECK OLD TRAINING STATE ===

from pathlib import Path
import zipfile
import shutil

zip_path = Path("/content/mira_user_voice_samples.zip")
voice_dir = Path("/content/mira_user_voice_samples")

assert zip_path.exists(), "mira_user_voice_samples.zip was not found."

# Extract the 52 real Mira recordings
if voice_dir.exists():
    shutil.rmtree(voice_dir)

voice_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(voice_dir)

wav_files = sorted(voice_dir.rglob("*.wav"))

print("=== REAL MIRA VOICE SAMPLES ===")
print("WAV FILES:", len(wav_files))

if wav_files:
    print("FIRST:", wav_files[0])
    print("LAST :", wav_files[-1])

print()
print("=== PREVIOUS MIRA TRAINING STATE ===")

checks = {
    "Best weights":
        Path("/content/trained_models/mira/best_weights.weights.h5"),

    "Last weights":
        Path("/content/trained_models/mira/last_weights.weights.h5"),

    "Synthetic features":
        Path("/content/generated_augmented_features"),

    "Confusable features":
        Path("/content/confusable_features"),

    "Training config":
        Path("/content/training_parameters.yaml"),

    "microWakeWord repo":
        Path("/content/microWakeWord"),
}

for name, path in checks.items():
    print(
        f"{name:25} :",
        "FOUND" if path.exists() else "MISSING"
    )

print()
print("VOICE_SAMPLES_READY =", len(wav_files) == 52)

=== REAL MIRA VOICE SAMPLES ===
WAV FILES: 52
FIRST: /content/mira_user_voice_samples/mira_real_001.wav
LAST : /content/mira_user_voice_samples/mira_real_052.wav

=== PREVIOUS MIRA TRAINING STATE ===
Best weights              : MISSING
Last weights              : MISSING
Synthetic features        : MISSING
Confusable features       : MISSING
Training config           : MISSING
microWakeWord repo        : MISSING

VOICE_SAMPLES_READY = True


In [3]:
# === CHECK GOOGLE DRIVE FOR SAVED MIRA TRAINING STATE ===

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

root = Path("/content/drive/MyDrive")
mira_drive = root / "wakeword_training_mira"

print("\n=== KNOWN MIRA DRIVE FOLDER ===")
print("Folder exists:", mira_drive.exists())
print("Path:", mira_drive)

if mira_drive.exists():
    print("\nContents:")
    for p in sorted(mira_drive.rglob("*")):
        if p.is_file():
            print(
                p.relative_to(mira_drive),
                f"({p.stat().st_size / 1024:.1f} KB)"
            )

print("\n=== SEARCHING DRIVE FOR IMPORTANT TRAINING FILES ===")

targets = {
    "best_weights.weights.h5",
    "last_weights.weights.h5",
    "training_parameters.yaml",
    "training_config.yaml",
    "mira.tflite",
    "mira.json",
}

found = []

for p in root.rglob("*"):
    if p.name in targets:
        found.append(p)

for p in found:
    print(p)

print("\nFOUND_COUNT =", len(found))

Mounted at /content/drive

=== KNOWN MIRA DRIVE FOLDER ===
Folder exists: True
Path: /content/drive/MyDrive/wakeword_training_mira

Contents:
_run_finished.txt (0.1 KB)
mira.json (0.3 KB)
mira.tflite (124.1 KB)

=== SEARCHING DRIVE FOR IMPORTANT TRAINING FILES ===
/content/drive/MyDrive/wakeword_training_mira/mira.tflite
/content/drive/MyDrive/wakeword_training_mira/mira.json

FOUND_COUNT = 2


In [4]:
# === MIRA V2 — SAVE REAL VOICE DATA PERMANENTLY ===

from pathlib import Path
import shutil

src_dir = Path("/content/mira_user_voice_samples")

drive_root = Path(
    "/content/drive/MyDrive/wakeword_training_mira_v2"
)

voice_out = drive_root / "real_voice"

drive_root.mkdir(parents=True, exist_ok=True)

if voice_out.exists():
    shutil.rmtree(voice_out)

shutil.copytree(
    src_dir,
    voice_out
)

wav_files = sorted(
    voice_out.rglob("*.wav")
)

print("=== MIRA V2 PERSISTENT DATA ===")
print("Drive folder:", drive_root)
print("Real voice samples:", len(wav_files))
print("Saved to:", voice_out)

if len(wav_files) != 52:
    raise RuntimeError(
        f"Expected 52 real Mira samples, found {len(wav_files)}"
    )

print()
print("MIRA_REAL_VOICE_SAVED = True")

=== MIRA V2 PERSISTENT DATA ===
Drive folder: /content/drive/MyDrive/wakeword_training_mira_v2
Real voice samples: 52
Saved to: /content/drive/MyDrive/wakeword_training_mira_v2/real_voice

MIRA_REAL_VOICE_SAVED = True


In [5]:
# === MIRA V2 — CONFIG + TRAINING ENVIRONMENT ===

import os
import sys
import subprocess
import importlib
import re
from pathlib import Path

# ------------------------------------------------------------
# Mira v2 configuration
# ------------------------------------------------------------

WAKE_WORD = "Mira"
OUTPUT_NAME = "mira_v2"

AUTHOR = "Ali"
AUTHOR_WEBSITE = ""

DRIVE_FOLDER = "wakeword_training_mira_v2"
DRIVE_DIR = f"/content/drive/MyDrive/{DRIVE_FOLDER}"

WAKE_WORD_IPA_US = "ˈmiːɹə"
WAKE_WORD_IPA_UK = "ˈmiːrə"

# Smaller synthetic set than v1 because we now have your real voice
SAMPLES_US = 8000
SAMPLES_UK = 4000

CONFUSABLE_PHRASES = [
    "Mia",
    "Mila",
    "Mina",
    "Myra",
    "Kira",
    "Vera",
    "mirror",
    "Miranda",
    "miracle",
    "nearer",
    "Siri",
    "Alexa",
    "Gemini",
    "hey Google",
    "okay Google",
    "okay Nabu",
]

SAMPLES_PER_CONFUSABLE = 400

PROBABILITY_CUTOFF = 0.85
SLIDING_WINDOW_SIZE = 5
TENSOR_ARENA_SIZE = 50000
TRAINED_LANGUAGES = ["en"]

REAL_VOICE_DIR = Path(
    "/content/drive/MyDrive/"
    "wakeword_training_mira_v2/real_voice"
)

real_wavs = sorted(REAL_VOICE_DIR.rglob("*.wav"))

print("=== MIRA V2 CONFIG ===")
print("Real voice samples:", len(real_wavs))
print("Synthetic positives planned:", SAMPLES_US + SAMPLES_UK)
print(
    "Confusable negatives planned:",
    len(CONFUSABLE_PHRASES) * SAMPLES_PER_CONFUSABLE
)

assert len(real_wavs) == 52, (
    f"Expected 52 real recordings, found {len(real_wavs)}"
)

os.makedirs(DRIVE_DIR, exist_ok=True)

# ------------------------------------------------------------
# Verify GPU
# ------------------------------------------------------------

try:
    import torch

    print()
    print("=== GPU ===")
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("GPU check warning:", e)

# ------------------------------------------------------------
# Install microWakeWord dependencies
# ------------------------------------------------------------

DEPS = [
    "audiomentations",
    "audio_metadata",
    "datasets",
    "mmap_ninja",
    "numpy",
    "pymicro-features",
    "pyyaml",
    "tensorflow>=2.16",
    "webrtcvad-wheels",
    "ai-edge-litert",
    (
        "git+https://github.com/whatsnowplaying/"
        "audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f"
    ),
]

print()
print("Installing training dependencies...")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + DEPS,
    check=True,
)

# ------------------------------------------------------------
# Clone microWakeWord
# ------------------------------------------------------------

os.chdir("/content")

if not Path("/content/microWakeWord").exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/kahrendt/microWakeWord",
            "/content/microWakeWord",
        ],
        check=True,
    )

MWW_DIR = "/content/microWakeWord"

if MWW_DIR not in sys.path:
    sys.path.insert(0, MWW_DIR)

importlib.invalidate_caches()

# ------------------------------------------------------------
# Patch newer TensorFlow numpy behavior
# ------------------------------------------------------------

train_py = Path(
    "/content/microWakeWord/microwakeword/train.py"
)

src = train_py.read_text()

patched = re.sub(
    r'(\b[a-zA-Z_]+\["[a-z]+"\])\.numpy\(\)',
    r'(\1.numpy() if hasattr(\1, "numpy") else \1)',
    src,
)

patch_count = (
    patched.count("hasattr") -
    src.count("hasattr")
)

if patch_count > 0:
    train_py.write_text(patched)
    print(
        f"Patched {patch_count} TensorFlow compatibility call(s)."
    )

# ------------------------------------------------------------
# Import check
# ------------------------------------------------------------

import microwakeword

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

print()
print("======================================")
print("MIRA V2 TRAINING ENVIRONMENT READY")
print("======================================")
print("microWakeWord:", MWW_DIR)
print("Real samples:", len(real_wavs))
print("Drive output:", DRIVE_DIR)

=== MIRA V2 CONFIG ===
Real voice samples: 52
Synthetic positives planned: 12000
Confusable negatives planned: 6400

=== GPU ===
CUDA available: True
GPU: Tesla T4

Installing training dependencies...

MIRA V2 TRAINING ENVIRONMENT READY
microWakeWord: /content/microWakeWord
Real samples: 52
Drive output: /content/drive/MyDrive/wakeword_training_mira_v2


In [8]:
# === MIRA V2 — INSTALL PIPER SAMPLE GENERATOR PROPERLY ===

import os
import sys
import subprocess
from pathlib import Path

os.chdir("/content")

# Install the maintained package itself
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "webrtcvad-wheels",
        "piper-tts==1.3.0",
        "piper-sample-generator",
    ],
    check=True,
)

print("Testing imports...")

test = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import piper_sample_generator; "
            "print('PIPER_SAMPLE_GENERATOR_OK')"
        ),
    ],
    capture_output=True,
    text=True,
)

print(test.stdout)

if test.stderr:
    print(test.stderr)

print("Exit code:", test.returncode)

if test.returncode != 0:
    raise RuntimeError(
        "piper_sample_generator still failed to import."
    )

print()
print("======================================")
print("PIPER SAMPLE GENERATOR READY")
print("======================================")

Testing imports...
PIPER_SAMPLE_GENERATOR_OK

Exit code: 0

PIPER SAMPLE GENERATOR READY


In [11]:
# === FIX PIPER monotonic_align — CORRECT BUILD COMMAND ===

import os
import sys
import subprocess
from pathlib import Path

align_dir = Path(
    "/content/piper/src/python/"
    "piper_train/vits/monotonic_align"
)

print("Alignment source:", align_dir)

if not align_dir.exists():
    raise RuntimeError(
        f"Missing folder: {align_dir}"
    )

# Build dependencies
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "cython",
        "setuptools",
        "wheel",
    ],
    check=True,
)

os.chdir(align_dir)

print()
print("Files before build:")
for p in sorted(align_dir.iterdir()):
    print(" ", p.name)

print()
print("Building with cythonize -i core.pyx ...")

proc = subprocess.run(
    [
        "cythonize",
        "-i",
        "core.pyx",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(proc.stdout)
print("Build exit code:", proc.returncode)

if proc.returncode != 0:
    raise RuntimeError(
        "monotonic_align build failed. "
        "Send me the full output above."
    )

print()
print("Searching for compiled extension...")

compiled = list(
    align_dir.rglob("*.so")
)

for f in compiled:
    print(" ", f)

if not compiled:
    raise RuntimeError(
        "Build returned success but no .so file was found."
    )

# Piper imports:
# piper_train.vits.monotonic_align.monotonic_align.core
target_dir = align_dir / "monotonic_align"
target_dir.mkdir(exist_ok=True)

for so_file in compiled:
    if so_file.parent == target_dir:
        continue

    target = target_dir / so_file.name

    if target.exists():
        target.unlink()

    so_file.replace(target)

print()
print("Final extension folder:")
for p in sorted(target_dir.iterdir()):
    print(" ", p.name)

os.chdir("/content")

env = os.environ.copy()

env["PYTHONPATH"] = ":".join(
    [
        "/content/piper/src/python",
        env.get("PYTHONPATH", ""),
    ]
)

print()
print("Testing Piper import...")

test = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "from piper_train.vits.monotonic_align "
            "import maximum_path; "
            "print('MONOTONIC_ALIGN_OK')"
        ),
    ],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(test.stdout)
print("Import exit code:", test.returncode)

if test.returncode != 0:
    raise RuntimeError(
        "Compiled extension exists, but import still failed."
    )

print()
print("======================================")
print("PIPER LEGACY EXTENSION READY")
print("======================================")

Alignment source: /content/piper/src/python/piper_train/vits/monotonic_align

Files before build:
  Makefile
  __init__.py
  __pycache__
  core.c
  core.pyx
  setup.py

Building with cythonize -i core.pyx ...
performance hint: core.pyx:5:0: Exception check on 'maximum_path_each' will always require the GIL to be acquired.
Possible solutions:
	1. Declare 'maximum_path_each' as 'noexcept' if you control the definition and you're sure you don't want the function to raise exceptions.
	2. Use an 'int' return type on 'maximum_path_each' to allow an error code to be returned.
performance hint: core.pyx:36:0: Exception check on 'maximum_path_c' will always require the GIL to be acquired.
Possible solutions:
	1. Declare 'maximum_path_c' as 'noexcept' if you control the definition and you're sure you don't want the function to raise exceptions.
	2. Use an 'int' return type on 'maximum_path_c' to allow an error code to be returned.
performance hint: core.pyx:42:21: Exception check after calling '

In [15]:
# === MIRA V2 — CREATE LIBRITTS-R GENERATOR CONFIG + TEST ===

import json
import os
import sys
import shutil
import subprocess
from pathlib import Path

model_path = Path("/content/models/en_US-libritts_r-medium.pt")
config_path = Path(str(model_path) + ".json")
test_dir = Path("/content/mira_v2_test")

assert model_path.exists(), f"Missing model: {model_path}"

# Minimal config required by piper_sample_generator's .pt path
config = {
    "espeak": {
        "voice": "en-us"
    },
    "audio": {
        "sample_rate": 22050
    },
    "num_speakers": 904
}

config_path.write_text(
    json.dumps(config, indent=2),
    encoding="utf-8"
)

print("CONFIG CREATED:")
print(config_path)
print(config_path.read_text())

env = os.environ.copy()

env["PYTHONPATH"] = ":".join([
    "/content/piper/src/python",
    "/content/piper-sample-generator",
    env.get("PYTHONPATH", ""),
])

if test_dir.exists():
    shutil.rmtree(test_dir)

test_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    "-m",
    "piper_sample_generator",
    "Mira",
    "--model",
    str(model_path),
    "--max-samples",
    "10",
    "--batch-size",
    "8",
    "--max-speakers",
    "50",
    "--length-scales",
    "0.9",
    "1.0",
    "1.1",
    "--output-dir",
    str(test_dir),
]

print()
print("======================================")
print("RUNNING 10-SAMPLE MIRA TEST")
print("======================================")
print()

proc = subprocess.Popen(
    cmd,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end="")

proc.wait()

wavs = sorted(test_dir.glob("*.wav"))

print()
print("Exit code:", proc.returncode)
print("TEST WAV FILES:", len(wavs))

if proc.returncode != 0:
    raise RuntimeError(
        "Mira 10-sample generation test failed."
    )

if len(wavs) != 10:
    raise RuntimeError(
        f"Expected 10 WAV files, found {len(wavs)}."
    )

print()
print("======================================")
print("MIRA 10-SAMPLE TEST PASSED")
print("======================================")

CONFIG CREATED:
/content/models/en_US-libritts_r-medium.pt.json
{
  "espeak": {
    "voice": "en-us"
  },
  "audio": {
    "sample_rate": 22050
  },
  "num_speakers": 904
}

RUNNING 10-SAMPLE MIRA TEST

DEBUG:__main__:Loading /content/models/en_US-libritts_r-medium.pt
INFO:__main__:Successfully loaded the model
DEBUG:__main__:CUDA available, using GPU
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.13/dist-packages/piper_sample_generator/__main__.py", line 628, in <module>
    main()
    ~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/piper_sample_generator/__main__.py", line 619, in main
    generate_samples(**args)
    ~~~~~~~~~~~~~~~~^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/piper_sample_generator/__main__.py", line 144, in generate_samples
    phoneme_ids = get_phonemes(
        voice, config, next(texts), verbose, phoneme_input
    )
  Fi

RuntimeError: Mira 10-sample generation test failed.

In [9]:
# === MIRA V2 — DOWNLOAD LIBRITTS GENERATOR + GENERATE DATA ===

import os
import sys
import shutil
import subprocess
from pathlib import Path

os.chdir("/content")

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

models_dir = Path("/content/models")
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "en_US-libritts_r-medium.pt"

positive_dir = Path("/content/generated_samples_v2")
confusable_root = Path("/content/confusable_negatives_v2")

drive_data_dir = Path(
    "/content/drive/MyDrive/"
    "wakeword_training_mira_v2/generated_data"
)

drive_data_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Download LibriTTS-R generator
# ------------------------------------------------------------

if not model_path.exists():
    print("Downloading LibriTTS-R generator...")

    subprocess.run(
        [
            "wget",
            "-q",
            "--show-progress",
            "-O",
            str(model_path),
            (
                "https://github.com/rhasspy/"
                "piper-sample-generator/releases/download/"
                "v2.0.0/en_US-libritts_r-medium.pt"
            ),
        ],
        check=True,
    )

print()
print(
    "MODEL:",
    model_path,
    f"{model_path.stat().st_size / 1024 / 1024:.1f} MB"
)

# ------------------------------------------------------------
# Make sure the generator can see piper_train
# ------------------------------------------------------------

env = os.environ.copy()

env["PYTHONPATH"] = ":".join(
    [
        "/content/piper/src/python",
        "/content/piper-sample-generator",
        env.get("PYTHONPATH", ""),
    ]
)

# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

def generate(text, count, out_dir, batch_size=32):
    out_dir = Path(out_dir)

    if out_dir.exists():
        shutil.rmtree(out_dir)

    out_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable,
        "-m",
        "piper_sample_generator",
        text,
        "--model",
        str(model_path),
        "--max-samples",
        str(count),
        "--batch-size",
        str(batch_size),
        "--max-speakers",
        "400",
        "--length-scales",
        "0.75",
        "0.9",
        "1.0",
        "1.15",
        "1.3",
        "--slerp-weights",
        "0.25",
        "0.5",
        "0.75",
        "--output-dir",
        str(out_dir),
    ]

    print()
    print("=" * 60)
    print(f'Generating "{text}"')
    print("Samples:", count)
    print("Output :", out_dir)
    print("=" * 60)

    proc = subprocess.Popen(
        cmd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in proc.stdout:
        print(line, end="")

    proc.wait()

    if proc.returncode != 0:
        raise RuntimeError(
            f'Generation failed for "{text}"'
        )

    wavs = list(out_dir.glob("*.wav"))

    print()
    print(
        f'Finished "{text}":',
        len(wavs),
        "WAV files"
    )

    if len(wavs) != count:
        raise RuntimeError(
            f'Expected {count} samples for "{text}", '
            f'found {len(wavs)}'
        )

# ------------------------------------------------------------
# Generate Mira positives
# ------------------------------------------------------------

# We're using one 12k pool rather than separate US/UK directories.
# Speaker diversity + speed variation supplies the variety here.

generate(
    "Mira",
    12000,
    positive_dir,
    batch_size=32,
)

# ------------------------------------------------------------
# Generate confusables
# ------------------------------------------------------------

CONFUSABLE_PHRASES = [
    "Mia",
    "Mila",
    "Mina",
    "Myra",
    "Kira",
    "Vera",
    "mirror",
    "Miranda",
    "miracle",
    "nearer",
    "Siri",
    "Alexa",
    "Gemini",
    "hey Google",
    "okay Google",
    "okay Nabu",
]

confusable_root.mkdir(
    parents=True,
    exist_ok=True,
)

for i, phrase in enumerate(
    CONFUSABLE_PHRASES,
    start=1
):
    safe = (
        phrase.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    out_dir = confusable_root / safe

    print()
    print(
        f"[{i}/{len(CONFUSABLE_PHRASES)}]"
    )

    generate(
        phrase,
        400,
        out_dir,
        batch_size=32,
    )

# ------------------------------------------------------------
# Flatten confusables with unique filenames
# ------------------------------------------------------------

flat_confusables = Path(
    "/content/confusable_negatives_v2_flat"
)

if flat_confusables.exists():
    shutil.rmtree(flat_confusables)

flat_confusables.mkdir(
    parents=True,
    exist_ok=True,
)

counter = 0

for folder in sorted(
    confusable_root.iterdir()
):
    if not folder.is_dir():
        continue

    for wav in sorted(folder.glob("*.wav")):
        counter += 1

        destination = (
            flat_confusables /
            f"confusable_{counter:05d}.wav"
        )

        shutil.copy2(
            wav,
            destination,
        )

# ------------------------------------------------------------
# Final count
# ------------------------------------------------------------

positive_count = len(
    list(positive_dir.glob("*.wav"))
)

negative_count = len(
    list(flat_confusables.glob("*.wav"))
)

print()
print("======================================")
print("MIRA V2 SYNTHETIC DATA COMPLETE")
print("======================================")
print("Positive Mira samples:", positive_count)
print("Confusable samples:   ", negative_count)
print("Real voice samples:   ", 52)

if positive_count != 12000:
    raise RuntimeError(
        f"Positive count wrong: {positive_count}"
    )

if negative_count != 6400:
    raise RuntimeError(
        f"Confusable count wrong: {negative_count}"
    )

# ------------------------------------------------------------
# Save raw generated data to Drive
# ------------------------------------------------------------

print()
print("Saving generated data to Drive...")

positive_drive = (
    drive_data_dir / "synthetic_mira"
)

negative_drive = (
    drive_data_dir / "confusables"
)

if positive_drive.exists():
    shutil.rmtree(positive_drive)

if negative_drive.exists():
    shutil.rmtree(negative_drive)

shutil.copytree(
    positive_dir,
    positive_drive,
)

shutil.copytree(
    flat_confusables,
    negative_drive,
)

print()
print("DRIVE BACKUP COMPLETE")
print("Positive:", positive_drive)
print("Negative:", negative_drive)


MODEL: /content/models/en_US-libritts_r-medium.pt 194.6 MB

Generating "Mira"
Samples: 12000
Output : /content/generated_samples_v2
DEBUG:__main__:Loading /content/models/en_US-libritts_r-medium.pt
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.13/dist-packages/piper_sample_generator/__main__.py", line 628, in <module>
    main()
    ~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/piper_sample_generator/__main__.py", line 619, in main
    generate_samples(**args)
    ~~~~~~~~~~~~~~~~^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/piper_sample_generator/__main__.py", line 74, in generate_samples
    torch_model = torch.load(model_path, weights_only=False)
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 1579, in load
    return _load(
        opened_zipfile,
    ...<3 lines>...
        **pickle_load_args,
    )
  File "

RuntimeError: Generation failed for "Mira"

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║                    CONFIGURE YOUR WAKE WORD HERE                       ║
# ╚═══════════════════════════════════════════════════════════════════════╝

# ─── Wake word identity ───
WAKE_WORD = "Hey Harold"           # Human-readable name (shown in HA)
OUTPUT_NAME = "hey_harold"         # Filename (no spaces, lowercase)
AUTHOR = "your_name"               # Goes in the manifest
AUTHOR_WEBSITE = "https://github.com/yourname"

# ─── Mode ───
MODE = "generate"   # "generate" (Piper makes samples) | "bundle" (you uploaded a zip)

# ─── Drive folder (created if missing) ───
DRIVE_FOLDER = "wakeword_training_hey_harold"

# ─── If MODE == "generate" ───
# Look up IPA for your wake word: https://www.internationalphoneticassociation.org/IPA-chart
# or use eSpeak: `espeak-ng -q --ipa "Hey Harold"` on Linux
WAKE_WORD_IPA_US = "hˈeɪ hˈærəld"   # US English pronunciation
WAKE_WORD_IPA_UK = "hˈeɪ hˈɛrəld"   # UK English (set to None to skip second pass)
SAMPLES_US = 30000                  # ~12 min on T4, ~7 min on A100
SAMPLES_UK = 15000                  # 0 to skip UK pass

# Confusable phrases — should NOT trigger your wake word.
# Include: phonetic neighbors, the bare name without prefix, common
# false-trigger phrases like other assistant names.
CONFUSABLE_PHRASES = [
    # General "hey X" near-misses
    "hey there", "hey you", "hey y'all", "hey now",
    # Other assistant wake words (must not steal yours)
    "hey siri", "hey google", "okay google", "hey alexa", "okay nabu",
    # WAKE-WORD SPECIFIC — replace these for your word!
    "hey howard", "hey harvey", "hey gerald", "hey carol",  # H-name neighbors
    "harold", "the herald",                                  # bare name + similar
    "hairy old", "hello harold",                              # phonetic mash-ups
]
SAMPLES_PER_CONFUSABLE = 1000   # 500 = light, 1000 = recommended, 2000 = max

# ─── If MODE == "bundle" ───
# Expected: <DRIVE_FOLDER>/data_bundle.zip with this layout:
#   generated_samples/        positive samples (.wav, 16 kHz mono)
#   real_recordings/          (optional) real mic recordings
#   confusable_negatives/     (optional) hard negative samples
BUNDLE_NAME = "data_bundle.zip"

# ─── Manifest tuning (works well as defaults; tune after on-device testing) ───
PROBABILITY_CUTOFF = 0.85          # 0.5 = too lenient, 0.97 = okay_nabu strict
SLIDING_WINDOW_SIZE = 5            # 5 = okay_nabu default; higher = slower fire
TENSOR_ARENA_SIZE = 50000          # 30000 works, 50000 has more headroom

# ─── Languages this model is trained for (manifest metadata) ───
TRAINED_LANGUAGES = ["en"]         # add "it", "es" etc if you have multi-accent samples

# ╔═══════════════════════════════════════════════════════════════════════╗
# ║                           END OF CONFIG                                ║
# ╚═══════════════════════════════════════════════════════════════════════╝
print(f"Training '{WAKE_WORD}' as {OUTPUT_NAME} in mode={MODE!r}")
print(f"Output → /content/drive/MyDrive/{DRIVE_FOLDER}/{OUTPUT_NAME}.tflite")


In [ ]:
# === Mount Drive ===
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE_DIR = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive folder: {DRIVE_DIR}')
if MODE == 'bundle':
    BUNDLE_PATH = f'{DRIVE_DIR}/{BUNDLE_NAME}'
    assert os.path.exists(BUNDLE_PATH), (
        f'MODE=bundle but {BUNDLE_PATH} does not exist. Upload your data zip there.')
    print(f'Found bundle: {os.path.getsize(BUNDLE_PATH)/1024/1024:.1f} MB')


In [ ]:
# === Install microWakeWord (kernel-restart-free) ===
# Workarounds for two upstream bugs:
#  1. kahrendt/microWakeWord setup.py has no find_packages() — non-editable
#     install skips the audio/ subpackage. Editable install needs kernel
#     restart, breaks Run All. Fix: install deps + sys.path.insert().
#  2. train.py calls .numpy() on values that newer TF returns as numpy
#     arrays already. Patch with hasattr() guard.
import os, sys, subprocess, importlib, re

DEPS = [
    'audiomentations', 'audio_metadata', 'datasets', 'mmap_ninja', 'numpy',
    'pymicro-features', 'pyyaml', 'tensorflow>=2.16', 'webrtcvad-wheels',
    'ai-edge-litert',
    'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f',
]
print('Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + DEPS, check=True)

if not os.path.exists('microWakeWord'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/kahrendt/microWakeWord'], check=True)

MWW_DIR = '/content/microWakeWord'
if MWW_DIR not in sys.path:
    sys.path.insert(0, MWW_DIR)
importlib.invalidate_caches()

fp = '/content/microWakeWord/microwakeword/train.py'
src = open(fp).read()
patched = re.sub(
    r'(\b[a-zA-Z_]+\["[a-z]+"\])\.numpy\(\)',
    r'(\1.numpy() if hasattr(\1, "numpy") else \1)',
    src
)
n = patched.count('hasattr') - src.count('hasattr')
if n > 0:
    open(fp, 'w').write(patched)
    print(f'Patched {n} .numpy() calls in train.py')

import microwakeword
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
print('OK: microwakeword.audio.* imports clean')


In [ ]:
# === Data preparation ===
# Mode-aware: either unzip user's bundle, or generate samples inline via Piper.
import os, zipfile

os.chdir('/content')

if MODE == 'bundle':
    print(f'Extracting {BUNDLE_PATH}...')
    with zipfile.ZipFile(BUNDLE_PATH, 'r') as zf:
        zf.extractall('/content')
    for d in ['generated_samples', 'real_recordings', 'confusable_negatives']:
        p = f'/content/{d}'
        if os.path.exists(p):
            n = sum(1 for _ in os.scandir(p) if _.name.endswith('.wav'))
            print(f'  {d}: {n} WAVs')
        else:
            print(f'  {d}: MISSING (will train without)')

elif MODE == 'generate':
    # Defer to the Piper sample-gen cells below
    print('MODE=generate — Piper sample-gen cells will produce samples')

else:
    raise ValueError(f'Unknown MODE: {MODE!r}. Use "bundle" or "generate".')


In [ ]:
# === Piper sample generator install (skipped if MODE=bundle) ===
if MODE == 'generate':
    import glob, os, shutil, subprocess, sys, urllib.request
    PIPER_REPO_DIR = '/content/piper'
    PIPER_SAMPLE_GENERATOR_DIR = '/content/piper-sample-generator'

    subprocess.run(['apt-get', '-qq', 'install', '-y', 'espeak-ng'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                    'pip', 'setuptools', 'wheel', 'cython'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                    'piper-tts', 'piper-sample-generator'], check=True)

    if not os.path.exists(PIPER_REPO_DIR):
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/rhasspy/piper', PIPER_REPO_DIR], check=True)
    if not os.path.exists(PIPER_SAMPLE_GENERATOR_DIR):
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/rhasspy/piper-sample-generator',
                        PIPER_SAMPLE_GENERATOR_DIR], check=True)

    PIPER_PYTHON_DIR = f'{PIPER_REPO_DIR}/src/python'
    MA_DIR = f'{PIPER_PYTHON_DIR}/piper_train/vits/monotonic_align'
    MA_IMPORT_DIR = f'{MA_DIR}/monotonic_align'
    MA_BUILD_DIR = f'{MA_DIR}/piper_train/vits/monotonic_align'

    shutil.rmtree(f'{PIPER_PYTHON_DIR}/build', ignore_errors=True)
    shutil.rmtree(MA_IMPORT_DIR, ignore_errors=True)
    shutil.rmtree(f'{MA_DIR}/piper_train', ignore_errors=True)
    os.makedirs(MA_IMPORT_DIR, exist_ok=True)
    os.makedirs(MA_BUILD_DIR, exist_ok=True)
    open(f'{MA_IMPORT_DIR}/__init__.py', 'a').close()
    subprocess.run(f'cd {MA_DIR} && {sys.executable} setup.py build_ext --inplace',
                   shell=True, check=True)
    built = next(iter(glob.glob(f'{MA_BUILD_DIR}/core.*')), None)
    assert built, 'monotonic_align core extension build failed'
    shutil.copy2(built, MA_IMPORT_DIR)

    for path in (PIPER_PYTHON_DIR, PIPER_SAMPLE_GENERATOR_DIR):
        if path not in sys.path:
            sys.path.insert(0, path)

    MODEL_PATH = 'models/en_US-libritts_r-medium.pt'
    MODEL_CONFIG_PATH = f'{MODEL_PATH}.json'
    os.makedirs('models', exist_ok=True)
    if not os.path.exists(MODEL_PATH):
        print('Downloading libritts_r model (~75 MB)...')
        urllib.request.urlretrieve(
            'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt',
            MODEL_PATH)
        urllib.request.urlretrieve(
            'https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/libritts_r/medium/en_US-libritts_r-medium.onnx.json',
            MODEL_CONFIG_PATH)
    print('Piper ready')
else:
    print('Skipped (MODE != generate)')


In [ ]:
# === Generate positive samples (US English, optionally UK) ===
if MODE == 'generate':
    import os, subprocess, sys
    PIPER_SAMPLE_GENERATOR_DIR = '/content/piper-sample-generator'
    MODEL_PATH = 'models/en_US-libritts_r-medium.pt'
    PIPER_BATCH = 256  # drop to 128 on T4 if CUDA OOM

    def run_piper(target_word, max_samples, output_dir):
        cmd = [sys.executable, f'{PIPER_SAMPLE_GENERATOR_DIR}/generate_samples.py',
               target_word, '--phoneme-input', '--model', MODEL_PATH,
               '--max-samples', str(max_samples),
               '--batch-size', str(PIPER_BATCH),
               '--noise-scales', '0.5', '--noise-scale-ws', '0.6',
               '--output-dir', output_dir]
        subprocess.run(cmd, check=True)

    os.makedirs('generated_samples', exist_ok=True)

    print(f'Generating {SAMPLES_US} US samples for {WAKE_WORD_IPA_US!r}...')
    run_piper(WAKE_WORD_IPA_US, SAMPLES_US, 'generated_samples')

    if WAKE_WORD_IPA_UK and SAMPLES_UK > 0:
        print(f'Generating {SAMPLES_UK} UK samples for {WAKE_WORD_IPA_UK!r}...')
        run_piper(WAKE_WORD_IPA_UK, SAMPLES_UK, 'generated_samples')

    n = sum(1 for f in os.listdir('generated_samples') if f.endswith('.wav'))
    print(f'Total positive samples: {n}')
else:
    print('Skipped (MODE != generate)')


In [ ]:
# === Generate confusable negatives ===
if MODE == 'generate':
    import os, subprocess, sys
    from pathlib import Path
    PIPER_SAMPLE_GENERATOR_DIR = '/content/piper-sample-generator'
    MODEL_PATH = 'models/en_US-libritts_r-medium.pt'

    os.makedirs('confusable_negatives', exist_ok=True)
    for phrase in CONFUSABLE_PHRASES:
        safe = phrase.replace(' ', '_').replace(',', '').replace("'", '')
        existing = len(list(Path('confusable_negatives').glob(f'{safe}_*.wav')))
        if existing >= SAMPLES_PER_CONFUSABLE:
            print(f'  {phrase!r}: {existing} already, skip')
            continue
        tmp = f'/tmp/confusable_{safe}'
        os.makedirs(tmp, exist_ok=True)
        print(f'  generating {SAMPLES_PER_CONFUSABLE} for {phrase!r}...')
        subprocess.run([sys.executable, f'{PIPER_SAMPLE_GENERATOR_DIR}/generate_samples.py',
                        phrase, '--model', MODEL_PATH,
                        '--max-samples', str(SAMPLES_PER_CONFUSABLE),
                        '--batch-size', '256',
                        '--noise-scales', '0.5', '--noise-scale-ws', '0.6',
                        '--output-dir', tmp], check=True)
        # Move + rename with safe prefix
        for f in os.listdir(tmp):
            if f.endswith('.wav'):
                os.rename(f'{tmp}/{f}', f'confusable_negatives/{safe}_{f}')

    n = sum(1 for f in os.listdir('confusable_negatives') if f.endswith('.wav'))
    print(f'Total confusable negatives: {n}')
else:
    print('Skipped (MODE != generate)')


In [ ]:
# === Download standard negative datasets (DNS challenge, AudioSet, MIT IRs) ===
# Pre-generated spectrogram features hosted on HuggingFace by kahrendt.
import os, subprocess
from huggingface_hub import snapshot_download

snapshot_download(
    'kahrendt/microwakeword',
    repo_type='dataset',
    local_dir='/content/negative_datasets',
    allow_patterns=['speech/*', 'dinner_party/*', 'no_speech/*',
                    'dinner_party_eval/*'],
)

# MIT room impulse responses for reverb augmentation
if not os.path.exists('mit_rirs') or not os.listdir('mit_rirs'):
    print('Downloading MIT room impulse responses...')
    subprocess.run(['mkdir', '-p', 'mit_rirs'], check=True)
    subprocess.run('cd mit_rirs && wget -q https://www.openslr.org/resources/28/rirs_noises.zip && unzip -q rirs_noises.zip',
                   shell=True, check=True)

# Background noise corpora (FMA + AudioSet 16 kHz subsets)
if not os.path.exists('fma_16k') or not os.listdir('fma_16k'):
    print('Downloading FMA background corpus (~500 MB)...')
    subprocess.run('mkdir -p fma_16k && cd fma_16k && wget -q https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/fma_16k.tar && tar -xf fma_16k.tar && rm fma_16k.tar',
                   shell=True, check=True)
if not os.path.exists('audioset_16k') or not os.listdir('audioset_16k'):
    print('Downloading AudioSet background corpus (~500 MB)...')
    subprocess.run('mkdir -p audioset_16k && cd audioset_16k && wget -q https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/audioset_16k.tar && tar -xf audioset_16k.tar && rm audioset_16k.tar',
                   shell=True, check=True)
print('All negative datasets ready')


In [ ]:
# === Augmentation + feature extraction ===
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
import os, shutil, traceback
from mmap_ninja.ragged import RaggedMmap

clips = Clips(
    input_directory='generated_samples',
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=True,
    random_split_seed=42,
    split_count=0.1,
)
augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.15, 'TanhDistortion': 0.10,
        'PitchShift': 0.15, 'BandStopFilter': 0.10,
        'AddColorNoise': 0.20, 'AddBackgroundNoise': 0.85,
        'Gain': 1.00, 'GainTransition': 0.25, 'RIR': 0.60,
    },
    impulse_paths=['mit_rirs'],
    background_paths=['fma_16k', 'audioset_16k'],
    background_min_snr_db=-5, background_max_snr_db=20,
    min_jitter_s=0.10, max_jitter_s=0.50,
)

os.makedirs('generated_augmented_features', exist_ok=True)
SPLIT_CONFIG = {
    'training':   {'split_name': 'train',      'repetition': 3, 'slide_frames': 10},
    'validation': {'split_name': 'validation', 'repetition': 1, 'slide_frames': 10},
    'testing':    {'split_name': 'test',       'repetition': 1, 'slide_frames': 1 },
}

for split, cfg in SPLIT_CONFIG.items():
    out = f'generated_augmented_features/{split}'
    mmap = f'{out}/wakeword_mmap'
    if os.path.exists(mmap) and list(os.scandir(mmap)):
        print(f'{split}: cached, skipping')
        continue
    if os.path.exists(mmap):
        shutil.rmtree(mmap)
    os.makedirs(out, exist_ok=True)
    print(f'Generating {split} (rep={cfg["repetition"]}, slide={cfg["slide_frames"]})...')
    try:
        sg = SpectrogramGeneration(clips=clips, augmenter=augmenter,
                                    slide_frames=cfg['slide_frames'], step_ms=10)
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    except Exception:
        traceback.print_exc()
        if os.path.exists(mmap): shutil.rmtree(mmap)
        raise
print('Positive features ready')

# Confusable features (only if confusable_negatives/ exists)
if os.path.exists('confusable_negatives') and os.listdir('confusable_negatives'):
    print('Generating confusable features...')
    confusable_clips = Clips(
        input_directory='confusable_negatives', file_pattern='*.wav',
        max_clip_duration_s=None, remove_silence=True,
        random_split_seed=42, split_count=0.1,
    )
    os.makedirs('confusable_features', exist_ok=True)
    for split, cfg in SPLIT_CONFIG.items():
        out = f'confusable_features/{split}'
        mmap = f'{out}/wakeword_mmap'
        if os.path.exists(mmap) and list(os.scandir(mmap)):
            continue
        if os.path.exists(mmap): shutil.rmtree(mmap)
        os.makedirs(out, exist_ok=True)
        sg = SpectrogramGeneration(clips=confusable_clips, augmenter=augmenter,
                                    slide_frames=cfg['slide_frames'], step_ms=10)
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    print('Confusable features ready')

# Real recording features (only if real_recordings/ exists)
if os.path.exists('real_recordings') and os.listdir('real_recordings'):
    print('Generating real-recording features...')
    real_clips = Clips(
        input_directory='real_recordings', file_pattern='*.wav',
        max_clip_duration_s=None, remove_silence=True,
        random_split_seed=42, split_count=0.1,
    )
    os.makedirs('real_recording_features', exist_ok=True)
    for split, cfg in SPLIT_CONFIG.items():
        out = f'real_recording_features/{split}'
        mmap = f'{out}/wakeword_mmap'
        if os.path.exists(mmap) and list(os.scandir(mmap)):
            continue
        if os.path.exists(mmap): shutil.rmtree(mmap)
        os.makedirs(out, exist_ok=True)
        sg = SpectrogramGeneration(clips=real_clips, augmenter=augmenter,
                                    slide_frames=cfg['slide_frames'], step_ms=10)
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    print('Real-recording features ready')


In [ ]:
# === Training config YAML ===
import yaml, os
from pathlib import Path

SKIP_CONFUSABLES = not Path('confusable_features/training/wakeword_mmap').exists()
SKIP_REAL = not Path('real_recording_features/training/wakeword_mmap').exists()

config = {
    'window_step_ms': 10,
    'train_dir': f'trained_models/{OUTPUT_NAME}',
    'features': [
        dict(features_dir='generated_augmented_features', sampling_weight=8.0,
             penalty_weight=2.0, truth=True, truncation_strategy='truncate_start',
             type='mmap'),
        dict(features_dir='negative_datasets/speech', sampling_weight=10.0,
             penalty_weight=2.5, truth=False, truncation_strategy='random', type='mmap'),
        dict(features_dir='negative_datasets/dinner_party', sampling_weight=15.0,
             penalty_weight=3.0, truth=False, truncation_strategy='random', type='mmap'),
        dict(features_dir='negative_datasets/no_speech', sampling_weight=5.0,
             penalty_weight=1.0, truth=False, truncation_strategy='random', type='mmap'),
        dict(features_dir='negative_datasets/dinner_party_eval', sampling_weight=0.0,
             penalty_weight=1.0, truth=False, truncation_strategy='split', type='mmap'),
    ],
    'training_steps': [25000, 20000],
    'positive_class_weight': [2, 2],
    'negative_class_weight': [40, 50],
    'learning_rates': [0.001, 0.0001],
    'batch_size': 256,
    'time_mask_max_size': [5, 5], 'time_mask_count': [1, 1],
    'freq_mask_max_size': [3, 3], 'freq_mask_count': [1, 1],
    'eval_step_interval': 500,
    'clip_duration_ms': 1500,
    'target_minimization': 0.4,
    'minimization_metric': 'ambient_false_positives_per_hour',
    'maximization_metric': 'average_viable_recall',
}

if not SKIP_CONFUSABLES:
    config['features'].append(dict(features_dir='confusable_features', sampling_weight=8.0,
                                    penalty_weight=5.0, truth=False,
                                    truncation_strategy='random', type='mmap'))
if not SKIP_REAL:
    config['features'].append(dict(features_dir='real_recording_features', sampling_weight=8.0,
                                    penalty_weight=2.0, truth=True,
                                    truncation_strategy='truncate_start', type='mmap'))

os.makedirs(f'trained_models/{OUTPUT_NAME}', exist_ok=True)
with open('training_parameters.yaml', 'w') as f:
    yaml.dump(config, f)
print('training_parameters.yaml ready')
print(f'  Feature sets: {len(config["features"])}, total steps: {sum(config["training_steps"])}')


In [ ]:
# === Train the model ===
# subprocess + PYTHONPATH so it can find microwakeword (sys.path doesn't propagate)
import os, sys, subprocess, shutil
shutil.rmtree(f'trained_models/{OUTPUT_NAME}', ignore_errors=True)

env = os.environ.copy()
env['PYTHONPATH'] = '/content/microWakeWord:' + env.get('PYTHONPATH', '')
env['XLA_FLAGS'] = '--xla_gpu_autotune_level=0'

cmd = [
    sys.executable, '-m', 'microwakeword.model_train_eval',
    '--training_config', 'training_parameters.yaml',
    '--train', '1', '--restore_checkpoint', '0',
    '--test_tflite_streaming_quantized', '1',
    '--use_weights', 'best_weights',
    'mixednet',
    '--pointwise_filters', '64,64,64,64',
    '--repeat_in_block', '1, 1, 1, 1',
    '--mixconv_kernel_sizes', '[5], [7,11], [9,15], [23]',
    '--residual_connection', '0,0,0,0',
    '--first_conv_filters', '32',
    '--first_conv_kernel_size', '5',
    '--stride', '3',
]
print('Running:', ' '.join(cmd)); print()
proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(); print('Exit code:', proc.returncode)
assert proc.returncode == 0, 'training failed - see output above'


In [ ]:
# === Export + push to Drive ===
import os, json, shutil, datetime

tflite_src = f'trained_models/{OUTPUT_NAME}/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite'
assert os.path.exists(tflite_src), f'No model at {tflite_src}'

OUT_TFLITE = f'{OUTPUT_NAME}.tflite'
OUT_JSON = f'{OUTPUT_NAME}.json'
shutil.copy2(tflite_src, OUT_TFLITE)
print(f'wrote {OUT_TFLITE} ({os.path.getsize(OUT_TFLITE)/1024:.1f} KB)')

manifest = {
    'type': 'micro',
    'wake_word': WAKE_WORD,
    'author': AUTHOR,
    'website': AUTHOR_WEBSITE,
    'model': OUT_TFLITE,
    'trained_languages': TRAINED_LANGUAGES,
    'version': 2,
    'micro': {
        'probability_cutoff': PROBABILITY_CUTOFF,
        'feature_step_size': 10,
        'sliding_window_size': SLIDING_WINDOW_SIZE,
        'tensor_arena_size': TENSOR_ARENA_SIZE,
        'minimum_esphome_version': '2024.7.0',
    }
}
with open(OUT_JSON, 'w') as f:
    json.dump(manifest, f, indent=2)
print(f'wrote {OUT_JSON}')
print(json.dumps(manifest, indent=2))

for fn in (OUT_TFLITE, OUT_JSON):
    shutil.copy2(fn, f'{DRIVE_DIR}/{fn}')
    print(f'pushed {fn} -> {DRIVE_DIR}')

ts = datetime.datetime.now(datetime.timezone.utc).isoformat()
with open(f'{DRIVE_DIR}/_run_finished.txt', 'w') as f:
    f.write(f'Training run finished at {ts}\n')
print()
print(f'DONE. Find your model at /content/drive/MyDrive/{DRIVE_FOLDER}/{OUTPUT_NAME}.tflite')


## Deploying to ESPHome devices

Drop the `.tflite` + `.json` next to your ESPHome YAML (e.g. in `/config/esphome/wakewords/<output_name>/`).

In your device YAML, replace your existing wake word:

```yaml
micro_wake_word:
  models:
    - model: wakewords/<output_name>/<output_name>.json
  on_wake_word_detected:
    - voice_assistant.start:
```

Then `esphome run <device>.yaml` (USB or OTA).

## Manifest tuning (likely needed)

The defaults work for **Hey Harold** specifically. Your model's confidence
distribution will differ. Iterate:

| Symptom | Knob |
|---|---|
| Doesn't fire on the wake word | Lower `probability_cutoff` (try 0.7, 0.6) |
| Fires on too many things | Raise `probability_cutoff` (try 0.92, 0.95) |
| LED fires but no STT response | `sliding_window_size: 5` (faster fire); also check Echo speaker mute |
| `Failed to allocate tensors` log | Raise `tensor_arena_size` (try 50000, 80000) |

## Known gotchas
- Editable install (`pip install -e ./microWakeWord`) requires kernel restart — broken for Run All
- `train.py` upstream calls `.numpy()` on numpy arrays under newer TF (this notebook patches it)
- T4 GPU OOMs during validation — use A100 + High-RAM
- Manifest path mismatch: training writes to `trained_models/<output_name>/`, manifest must match
- `voice_assistant` has no audio lookback; if MWW fires AFTER user speaks, STT gets silence
